# N02 — Regresi Linear & Logistik

**Pendamping S06 — Kecerdasan Buatan & ML Dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

## Kelompok C

**Nama Anggota:**
  - Dhika Muhammad Frizar (261635008)
  - Nabil Faiq Aflah (261636002)

## Bagian A — Linear Regression

## Eksperimen 1

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, classification_report
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(42)
X = rng.uniform(20, 120, size=(200,1))
y = 50 + 1.8*X[:,0] + rng.normal(0, 18, size=200)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)
print('intercept=', lin.intercept_, 'coef=', lin.coef_[0])
print('MAE=', mean_absolute_error(y_test,pred))
print('RMSE=', mean_squared_error(y_test,pred)**0.5)

intercept= 39.97840773813263 coef= 1.9237076136073439
MAE= 13.709271226328251
RMSE= 17.364824347784438


Interpretasikan koefisien tanpa mengklaim kausalitas. Cari 3 residual terbesar.

**Interpretasikan Koefisien**

Koefisien 1,92 artinya setiap kenaikan 1 unit fitur X, prediksi y naik sekitar 1,92 unit. Intercept 39,98 adalah prediksi y saat X = 0. Angka ini hanya titik awal garis, tidak bermakna nyata karena data X mulai dari 20. Ini hanya hubungan statistik, bukan sebab-akibat. Datanya juga sintetis (buatan), jadi tidak mewakili kondisi dunia nyata.

In [2]:
resid = y_test - pred
idx = np.argsort(np.abs(resid))[-3:]
pd.DataFrame({'aktual':y_test[idx], 'prediksi':pred[idx], 'residual':resid[idx]})

,aktual,prediksi,residual
0,199.721397,166.734537,32.986860
1,137.384165,98.344268,39.039897
2,265.371721,221.723023,43.648698


**Residual Terbesar**

Residual = aktual - prediksi. Tiga residual terbesar adalah 32,99; 39,04; dan 43,65. Ketiganya positif, artinya model memprediksi lebih rendah dari nilai aslinya. Ini wajar karena data dibuat dengan noise acak yang tidak bisa ditangkap garis lurus.

*note : hasil experimen 2 berada di paling bawah*

## Bagian B — Logistic Regression

## Eksperimen 3

In [3]:
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=.25, random_state=42, stratify=data.target
)
log = LogisticRegression(max_iter=5000).fit(X_train, y_train)
proba = log.predict_proba(X_test)[:,1]

for t in [0.3, 0.5, 0.7]:
    pred_t = (proba >= t).astype(int)
    print('\nthreshold', t)
    print(confusion_matrix(y_test,pred_t))
    print(classification_report(y_test,pred_t, digits=3))


threshold 0.3
[[46  7]
 [ 1 89]]
              precision    recall  f1-score   support

           0      0.979     0.868     0.920        53
           1      0.927     0.989     0.957        90

    accuracy                          0.944       143
   macro avg      0.953     0.928     0.938       143
weighted avg      0.946     0.944     0.943       143


threshold 0.5
[[48  5]
 [ 1 89]]
              precision    recall  f1-score   support

           0      0.980     0.906     0.941        53
           1      0.947     0.989     0.967        90

    accuracy                          0.958       143
   macro avg      0.963     0.947     0.954       143
weighted avg      0.959     0.958     0.958       143


threshold 0.7
[[49  4]
 [ 4 86]]
              precision    recall  f1-score   support

           0      0.925     0.925     0.925        53
           1      0.956     0.956     0.956        90

    accuracy                          0.944       143
   macro avg      0.940   

Bandingkan threshold 0.3, 0.5, 0.7. Apa yang berubah pada false positive dan false negative?

**Apa yang Berubah pada False Positive dan False Negative**

- Treshold 0.3: Model memprediksi "jinak" kalau probabilitasnya sudah ≥ 0.3. Syaratnya longgar, jadi lebih banyak data yang diprediksi jinak. Akibatnya False Positive naik jadi 7 (lebih banyak tumor ganas yang lolos jadi "jinak"), sedangkan False Negative turun jadi 1.
- Treshold 0.5: Titik tengah. False Positive 5 dan False Negative 1, dan akurasinya paling tinggi (0.958).
- Treshold 0.7: Model baru memprediksi "jinak" kalau probabilitasnya ≥ 0.7. Syaratnya ketat, jadi False Positive turun jadi 4, tapi False Negative naik jadi 4 (lebih banyak tumor jinak yang di prediksi ganas).

Jadi pola umumnya: **Treshold turun -> False Positive naik dan False Negative turun. Treshold naik -> False Positive turun dan False Negative naik.** Pada precision dan recall, threshold naik membuat precision naik (0.927 -> 0.956) dan recall turun (0.989 -> 0.956).

**Kenapa Ini Terjadi**

Logistic regression menghasilkan probabilitas lewat sigmoid, lalu threshold mengubah probabilitas itu menjadi label. Modelnya tetap sama, yang berubah hanya aturan keputusannya. Karena itu geser treshold selalu menimbulkan trade-off: Mengurangi satu jenis kesalahan akan menambah jenis kesalahan lainnya.

## Evidence Log

| Eksperimen | Perubahan | Metric / Output | Interpretasi |
|---|---|---|---|
| EXP-01 | Linear regression baseline, noise 18 | intercept 39.978; koef 1.923; MAE 13.709; RMSE 17.364 | Koefisien dekat nilai asli (1,8), tapi tidak persis karena ada noise. Error rata-rata sekitar 14 unit. |
| EXP-02 | Noise dinaikkan 18 → 40 | intercept 27.729; koef 2.074; MAE 30.465; RMSE 38.588 | Error naik sekitar 2x lipat. Koefisien ikut bergeser lebih jauh dari 1,8, jadi noise besar membuat model kurang stabil. |
| EXP-03 | Threshold 0,3 / 0,5 / 0,7 pada logistic regression | False Positive: 7 / 5 / 4. False Negative: 1 / 1 / 4. Akurasi: 0,944 / 0,958 / 0,944 | Threshold rendah: False Negative turun, False Positive naik. Threshold tinggi: False Positive turun, False Negative naik. Ada trade-off. |

## Refleksi akhir

1. Apa hasil utama yang Anda amati?
2. Mengapa hasil tersebut terjadi menurut konsep di Main Deck?
3. Apa limitation eksperimen ini?
4. Eksperimen apa yang layak dilakukan berikutnya?

**Jawaban**

1. Pada linear regression, model bisa menemukan hubungan antara X dan Y dengan koefisien 1.92, cukup dekat dengan nilai aslinya (1.8). Saat noise saya naikkan dari 18 ke 40, error ikut naik: MAE dari 13,71 jadi 30,47 dan RMSE dari 17,36 jadi 38,59. Pada logistic regression, saat threshold diubah dari 0,3 ke 0,7, FP turun dari 7 ke 4 dan FN naik dari 1 ke 4. Akurasi tertinggi ada di threshold 0,5 yaitu 0,958.
2. Linear regression mencari garis yang meminimalkan MSE. Kalau noise besar, titik data makin menyebar jauh dari garis, jadi residual dan error ikut naik. Pada logistic regression, sigmoid mengubah skor linear menjadi probabilitas, lalu threshold mengubah probabilitas itu menjadi label. Modelnya tidak berubah, hanya batas keputusannya yang bergeser. Karena itu mengurangi satu jenis kesalahan (False Positive) akan menambah jenis kesalahan lainnya (False Negative).
3.  - Data di Bagian A sintetis dan hanya punya satu fitur, jadi belum tentu mewakili data nyata.
    - Data uji kecil dan hanya satu kali split (random_state=42), sehingga selisih 1-2 data saja bisa mengubah hasil.
    - Fitur di Bagian B tidak di-scaling.
    - Koefisien hanya menunjukkan hubungan, bukan sebab-akibat.
4.  - Mencoba beberapa nilai random_state atau cross-validation agar hasil lebih stabil.
    - Menambahkan scaling (StandardScaler) pada logistic regression lalu membandingkan hasilnya.
    - Mencoba threshold lain seperti 0,4 dan 0,6.
    - Menambah fitur pada linear regression dan melihat apakah error turun.

## Eksperimen 2 (Bagian A)

Mengubah noise dari awalnya 18 menjadi 40, lalu bandingkan MAE, RMSE, dengan baseline, pada line ini **"y = 50 + 1.8*X[:,0] + rng.|normal(0, 40, size=200)",** yang sebelumnya seperti ini **"y = 50 + 1.8*X[:,0] + rng.normal(0, 18, size=200)".** Jadi di bagian A memiliki pembanding sendiri yaitu (noise 18 vs 40).

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, classification_report
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(42)
X = rng.uniform(20, 120, size=(200,1))
y = 50 + 1.8*X[:,0] + rng.normal(0, 40, size=200)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
lin = LinearRegression().fit(X_train, y_train)
pred = lin.predict(X_test)
print('intercept=', lin.intercept_, 'coef=', lin.coef_[0])
print('MAE=', mean_absolute_error(y_test,pred))
print('RMSE=', mean_squared_error(y_test,pred)**0.5)

intercept= 27.729794973627975 coef= 2.0749058080163207
MAE= 30.465047169618348
RMSE= 38.58849855063211
